# Banking loan default prediction with XGBoost
**Goal:** Predict `loan_default` from information available at loan application time. `0` means no default; `1` means default. This CSV contains 1,000 example records and 19 predictors. It includes numeric and text categories; the latter need encoding before XGBoost can use them.

**Run order:** Keep `Banking_Loan_Default_Classification(5).csv` beside this notebook and run the cells from top to bottom. The records are for demonstration, not a production credit policy.

This first notebook is a **basic baseline**: read, inspect, split, encode, fit, and evaluate.

## 1. Imports
`XGBClassifier` is the XGBoost classification model. Scikit-learn provides the train/test split, encoding pipeline, and evaluation metrics.

In [ ]:
# If packages are missing, run this once in a notebook cell, then restart the kernel:
# %pip install pandas numpy scikit-learn xgboost matplotlib seaborn joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score
from xgboost import XGBClassifier

## 2. Read and understand the data
Always check the target distribution. Here `loan_default=1` is the event we want to identify.

In [ ]:
df = pd.read_csv("Banking_Loan_Default_Classification(5).csv")
print("Rows and columns:", df.shape)
display(df.head())
print("Target counts (0=no default, 1=default):")
display(df["loan_default"].value_counts().sort_index())

## 3. Separate predictors and target; split the data
The test set is held aside before fitting the encoder. Stratification keeps roughly the same default proportion in both sets. `OneHotEncoder` turns text categories into 0/1 columns and ignores unseen categories during prediction.

In [ ]:
X = df.drop(columns="loan_default")
y = df["loan_default"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X_train.select_dtypes(include="number").columns.tolist()
print("Categorical columns:", cat_cols)
print("Numeric columns:", num_cols)
preprocessor = ColumnTransformer([
    ("categories", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("numbers", "passthrough", num_cols),
])

## 4. Fit a basic XGBoost model
A `Pipeline` fits the encoder on training records and then fits the model. No manual search or threshold tuning is used in this baseline. The fixed random seed makes this demonstration reproducible.

In [ ]:
basic_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", XGBClassifier(
        n_estimators=100, max_depth=3, learning_rate=0.1,
        objective="binary:logistic", eval_metric="logloss",
        random_state=42, n_jobs=2,
    )),
])
basic_model.fit(X_train, y_train)
print("Basic model fitted.")

## 5. Predict and interpret the results
At the default cutoff of 0.50, a probability of at least 0.50 becomes class 1. Accuracy counts all correct predictions; precision asks how many predicted defaults really defaulted; recall asks how many actual defaults were found; F1 balances precision and recall. ROC AUC evaluates ranking across many cutoffs.

In [ ]:
pred = basic_model.predict(X_test)
prob = basic_model.predict_proba(X_test)[:, 1]
print(f"Accuracy:  {accuracy_score(y_test, pred):.3f}")
print(f"Precision: {precision_score(y_test, pred, zero_division=0):.3f}")
print(f"Recall:    {recall_score(y_test, pred, zero_division=0):.3f}")
print(f"F1 score:  {f1_score(y_test, pred, zero_division=0):.3f}")
print(f"ROC AUC:   {roc_auc_score(y_test, prob):.3f}")
print("\nConfusion matrix: rows=actual, columns=predicted")
display(pd.DataFrame(confusion_matrix(y_test, pred, labels=[0, 1]),
                     index=["Actual 0", "Actual 1"],
                     columns=["Predicted 0", "Predicted 1"]))
print("\nClass-wise report:")
print(classification_report(y_test, pred, target_names=["No default", "Default"], zero_division=0))

## What next?
The second notebook adds checks for missing values, cross-validation, parameter tuning, a validation-only decision cutoff, readable visualizations, feature importance, saving, and prediction for a new application. Do not claim a particular accuracy before running the notebook: scores depend on this CSV and the fixed split.